In [ ]:
import pandas as pd

# load files

articles = pd.read_csv(
    "magyarnemzet_articles.csv",
    low_memory=False
)

high_recall = pd.read_csv(
    "magyarnemzet_ukraine_russia_high_recall.csv",
    low_memory=False
)

meta1 = pd.read_csv(
    "magyarnemzet_missing_metadata.csv",
    low_memory=False
)

meta2 = pd.read_csv(
    "magyarnemzet_missing_metadata_recovered.csv",
    low_memory=False
)


print("articles")
print(len(articles))

print()

print("high recall")
print(len(high_recall))

print()

print("metadata 1")
print(len(meta1))

print()

print("metadata 2")
print(len(meta2))

In [ ]:
import pandas as pd

# load files

second = pd.read_csv(
    "magyarnemzet_SECONDROUND_final_articles.csv",
    low_memory=False
)

full = pd.read_csv(
    "magyarnemzet_ukraine_russia_final.csv",
    low_memory=False
)

# keep only metadata columns

meta = full[
    [
        "article_url",
        "lead",
        "author",
        "main_image_url",
        "main_image_caption",
        "image_count",
        "all_image_urls",
        "all_image_captions"
    ]
].copy()

# remove duplicates

meta = meta.drop_duplicates(
    subset="article_url",
    keep="last"
)

# remove old metadata columns from the filtered corpus

second = second.drop(
    columns=[
        "lead",
        "author",
        "main_image_url",
        "main_image_caption",
        "image_count",
        "all_image_urls",
        "all_image_captions"
    ],
    errors="ignore"
)

# merge metadata back

second = second.merge(
    meta,
    on="article_url",
    how="left"
)



In [ ]:
# save


print("rows:", len(second))

print()

print(second.isna().sum())

In [ ]:
second.to_csv(
    "magyarnemzet_SECONDROUND_final_articles.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
print("main images:")
print(
    second["main_image_url"]
    .notna()
    .sum()
)

print()

print("article images:")
print(
    second["all_image_urls"]
    .notna()
    .sum()
)

In [ ]:
import pandas as pd
import ast
from collections import Counter

second = df

counter = Counter()

for _, row in df.iterrows():

    urls = set()

    if pd.notna(row["main_image_url"]):
        urls.add(row["main_image_url"].strip())

    if pd.notna(row["all_image_urls"]):

        try:
            imgs = ast.literal_eval(row["all_image_urls"])

            if isinstance(imgs, list):
                urls.update(
                    str(x).strip()
                    for x in imgs
                    if str(x).strip()
                )

        except:
            pass

    counter.update(urls)

print("unique images:", len(counter))

In [ ]:
freq = pd.DataFrame(
    counter.items(),
    columns=["image_url", "count"]
)

print(freq["count"].describe())

images = freq[
    freq["count"] <= 5
].copy()

print("images to download:", len(images))

In [ ]:
images

In [ ]:
import json
import pandas as pd
import requests
from tqdm import tqdm
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed
import traceback
# --------------------------------------------------
# files
# --------------------------------------------------

input_file = "magyarnemzet_SECONDROUND_final_articles.csv"
output_file = "magyarnemzet_SECONDROUND_final_articles_updated.csv"

workers = 10

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}

# --------------------------------------------------
# load
# --------------------------------------------------

df = pd.read_csv(input_file, low_memory=False)

mask = (
    df["main_image_url"].isna())

#todo = df.loc[mask, "article_url"].tolist()
todo = [
    "https://magyarnemzet.hu/belfold/2022/10/az-a-legfontosabb-hogy-a-magyar-csaladok-tudjanak-futeni-telen"
]
print("articles to update:", len(todo))


from urllib.parse import urlparse

def find_article(data, url):
    """
    Ищет именно ту статью, которая соответствует URL,
    а не первый попавшийся объект с body.
    """

    slug = urlparse(url).path.rstrip("/").split("/")[-1]

    best = None

    def walk(obj):
        nonlocal best

        if best is not None:
            return

        if isinstance(obj, dict):

            # --------------------------------------------------
            # Проверяем, похож ли объект на статью
            # --------------------------------------------------

            has_body = isinstance(obj.get("body"), list)

            if has_body:

                obj_slug = obj.get("slug")

                canonical = (
                    obj.get("canonicalUrl")
                    or obj.get("canonicalURL")
                    or obj.get("url")
                )

                # 1. Совпадает slug
                if obj_slug == slug:
                    best = obj
                    return

                # 2. Совпадает canonicalUrl
                if isinstance(canonical, str):

                    if canonical.rstrip("/") == url.rstrip("/"):
                        best = obj
                        return

                    if canonical.rstrip("/").endswith("/" + slug):
                        best = obj
                        return

            # продолжаем обход
            for value in obj.values():
                walk(value)

        elif isinstance(obj, list):

            for item in obj:
                walk(item)

    walk(data)

    # --------------------------------------------------
    # если нашли по slug
    # --------------------------------------------------

    if best is not None:
        return best

    # --------------------------------------------------
    # запасной вариант —
    # первый объект с body
    # --------------------------------------------------

    def fallback(obj):

        if isinstance(obj, dict):

            if isinstance(obj.get("body"), list):
                return obj

            for value in obj.values():

                result = fallback(value)

                if result is not None:
                    return result

        elif isinstance(obj, list):

            for item in obj:

                result = fallback(item)

                if result is not None:
                    return result

        return None

    return fallback(data)

def parse_article(url):

    try:

        r = requests.get(
            url,
            headers=headers,
            timeout=60
        )

        if r.status_code != 200:
            return {
                "article_url": url,
                "error": f"status_{r.status_code}"
            }

        soup = BeautifulSoup(r.text, "html.parser")

        state = soup.find("script", id="ng-state")

        if state is None:
            return {
                "article_url": url,
                "error": "missing_ng_state"
            }

        data = json.loads(state.text) 
       
        article = find_article(data, url)

        if article is None:
            return {
                "article_url": url,
                "error": "article_not_found"
            }

        # ---------- DEBUG ----------
        with open("article_debug.json", "w", encoding="utf-8") as f:
            json.dump(article, f, ensure_ascii=False, indent=2)

        print("saved article_debug.json")
        # ---------------------------

        author = article.get("publicAuthor")

        if isinstance(author, dict):
            author = author.get("fullName")

        lead = article.get("lead") or article.get("excerpt")

        main_image = (
            article.get("thumbnail")
            or article.get("metaThumbnail")
            or article.get("secondaryThumbnail")
        )

        caption = None

        info = article.get("thumbnailInfo")

        if isinstance(info, dict):
            caption = (
                info.get("caption")
                or info.get("title")
                or info.get("description")
            )

        image_urls = []
        image_captions = []

        body = article.get("body", [])

        if not isinstance(body, list):
            body = []

        for block in body:

            if not isinstance(block, dict):
                continue

            details = block.get("details", [])

            if not isinstance(details, list):
                continue

            for detail in details:

                if not isinstance(detail, dict):
                    continue

                html = detail.get("value")

                if not isinstance(html, str):
                    continue

                html_soup = BeautifulSoup(
                    html,
                    "html.parser"
                )

                # -------- figure/img --------

                for figure in html_soup.find_all("figure"):

                    img = figure.find("img")

                    if img and img.get("src"):

                        image_urls.append(img["src"])

                        figcaption = figure.find("figcaption")

                        if figcaption:
                            image_captions.append(
                                figcaption.get_text(
                                    " ",
                                    strip=True
                                )
                            )
                        else:
                            image_captions.append(None)

                # -------- просто img --------

                for img in html_soup.find_all("img"):

                    if img.get("src"):

                        src = img["src"]

                        if src not in image_urls:

                            image_urls.append(src)
                            image_captions.append(None)

        image_urls = list(dict.fromkeys(image_urls))

        if not main_image and image_urls:
            main_image = image_urls[0]

        return {

            "article_url": url,

            "author": author,

            "lead": lead,

            "main_image_url": main_image,

            "main_image_caption": caption,

            "image_count": len(image_urls),

            "all_image_urls": json.dumps(
                image_urls,
                ensure_ascii=False
            ),

            "all_image_captions": json.dumps(
                image_captions,
                ensure_ascii=False
            ),

            "error": None

        }

    except Exception as e:

        traceback.print_exc()

        return {
            "article_url": url,
            "error": str(e)
        }
# --------------------------------------------------
# crawl
# --------------------------------------------------

results = []

with ThreadPoolExecutor(max_workers=workers) as executor:

    futures = {
        executor.submit(parse_article, url): url
        for url in todo
    }

    for future in tqdm(
        as_completed(futures),
        total=len(futures),
        desc="recovering metadata"
    ):

        results.append(future.result())


# --------------------------------------------------
# results
# --------------------------------------------------

updates = pd.DataFrame(results)

print(updates["error"].value_counts(dropna=False))
print(updates[
    [
        "main_image_url",
        "image_count",
        "all_image_urls",
        "author",
        "lead"
    ]
].T)
print()
print("processed:", len(updates))

print()

print("successful:",
      updates["error"].isna().sum())

print()

print("failed:",
      updates["error"].notna().sum())

print()

if "error" in updates.columns:
    print(updates["error"].value_counts())


# --------------------------------------------------
# merge back
# --------------------------------------------------
# merge back
# --------------------------------------------------

good = updates.loc[
    updates["error"].isna()
].copy()

good = good.drop(columns="error")

df = df.set_index("article_url")
good = good.set_index("article_url")

columns = [
    "author",
    "lead",
    "main_image_url",
    "main_image_caption",
    "image_count",
    "all_image_urls",
    "all_image_captions"
]

# обновляем только существующие статьи
common_urls = df.index.intersection(good.index)

print("matched articles:", len(common_urls))

for col in columns:

    if col not in good.columns:
        continue

    # добавляем отсутствующий столбец
    if col not in df.columns:
        df[col] = pd.NA

    # обновляем только NaN
    mask = (
        df.loc[common_urls, col].isna()
        &
        good.loc[common_urls, col].notna()
    )

    df.loc[
        common_urls[mask],
        col
    ] = good.loc[
        common_urls[mask],
        col
    ]

df = df.reset_index()

# --------------------------------------------------
# save
# --------------------------------------------------

df.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print()
print("saved:", output_file)

print()

print(df[
    [
        "main_image_url",
        "all_image_urls"
    ]
].isna().sum())

## Ended up ignoring the missing files 

# Image download

In [ ]:
import os
import ast
import hashlib
from urllib.parse import urlsplit

import pandas as pd
import requests
from tqdm import tqdm

# --------------------------------------------------
# files
# --------------------------------------------------

input_file = "magyarnemzet_SECONDROUND_final_articles.csv"
output_file = "magyarnemzet_SECONDROUND_final_articles.csv"

image_dir = "magyarnemzet_SECONDROUND_images"

os.makedirs(image_dir, exist_ok=True)

headers = {
    "User-Agent": "Mozilla/5.0"
}

# --------------------------------------------------
# load
# --------------------------------------------------

df = pd.read_csv(
    input_file,
    low_memory=False
)

# --------------------------------------------------
# helpers
# --------------------------------------------------

def filename_from_url(url):

    path = urlsplit(url).path

    ext = os.path.splitext(path)[1].lower()

    if ext not in [
        ".jpg",
        ".jpeg",
        ".png",
        ".webp",
        ".gif",
        ".bmp",
        ".tif",
        ".tiff"
    ]:
        ext = ".jpg"

    h = hashlib.sha1(
        path.encode("utf-8")
    ).hexdigest()[:16]

    return h + ext


# --------------------------------------------------
# download
# --------------------------------------------------

downloaded = 0
skipped = 0
failed = 0

image_filenames = []

for _, row in tqdm(
    df.iterrows(),
    total=len(df)
):

    urls = []

    # main image

    if pd.notna(row["main_image_url"]):

        urls.append(
            row["main_image_url"]
        )

    # article images

    if pd.notna(row["all_image_urls"]):

        try:

            imgs = ast.literal_eval(
                row["all_image_urls"]
            )

            if isinstance(imgs, list):

                urls.extend(imgs)

        except:
            pass

    # remove duplicates inside article

    urls = list(dict.fromkeys(urls))

    filenames = []

    for url in urls:

        if not isinstance(url, str):
            continue

        url = url.strip()

        if not url:
            continue

        filename = filename_from_url(url)

        filenames.append(filename)

        filepath = os.path.join(
            image_dir,
            filename
        )

        if os.path.exists(filepath):

            skipped += 1
            continue

        try:

            r = requests.get(
                url,
                headers=headers,
                timeout=60
            )

            if r.status_code == 200:

                with open(
                    filepath,
                    "wb"
                ) as f:

                    f.write(r.content)

                downloaded += 1

            else:

                failed += 1

        except:

            failed += 1

    image_filenames.append(filenames)

# --------------------------------------------------
# save
# --------------------------------------------------

df["image_filenames"] = image_filenames

df.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print()
print("Downloaded:", downloaded)
print("Skipped:", skipped)
print("Failed:", failed)
print()
print("Saved:", output_file)

In [ ]:
df

In [ ]:
import pandas as pd
import ast
import numpy as np

def normalize_list(x):

    # NaN
    if isinstance(x, float) and pd.isna(x):
        return []

    # уже список
    if isinstance(x, list):
        return [str(i).strip() for i in x if str(i).strip()]

    # numpy array
    if isinstance(x, np.ndarray):
        return [str(i).strip() for i in x.tolist() if str(i).strip()]

    # строка
    if isinstance(x, str):

        x = x.strip()

        if not x:
            return []

        try:
            value = ast.literal_eval(x)

            if isinstance(value, list):
                return [str(i).strip() for i in value if str(i).strip()]
        except:
            pass

        return [x]

    return []

In [ ]:
image_urls = []

for _, row in df.iterrows():

    urls = []

    # main image
    if isinstance(row["main_image_url"], str):
        url = row["main_image_url"].strip()
        if url:
            urls.append(url)

    # article images
    urls.extend(normalize_list(row["all_image_urls"]))

    urls = list(dict.fromkeys(urls))

    image_urls.append("|".join(urls))

df["image_url"] = image_urls

In [ ]:
df["image_filename"] = (
    df["image_filenames"]
    .apply(normalize_list)
    .apply("|".join)
)

In [ ]:
print("image_url:",
      (df["image_url"].str.strip() == "").sum())

print("image_filename:",
      (df["image_filename"].str.strip() == "").sum())

In [ ]:
import pandas as pd
import ast

df = pd.read_csv(
    "magyarnemzet_SECONDROUND_final_articles.csv",
    low_memory=False
)

# --------------------------------------------------
# combine image urls
# --------------------------------------------------

combined_urls = []

for _, row in df.iterrows():

    urls = []

    # main image
    if pd.notna(row["main_image_url"]):

        url = str(row["main_image_url"]).strip()

        if url:
            urls.append(url)

    # article images
    if pd.notna(row["all_image_urls"]):

        try:

            imgs = ast.literal_eval(row["all_image_urls"])

            if isinstance(imgs, list):

                urls.extend(
                    str(x).strip()
                    for x in imgs
                    if str(x).strip()
                )

        except:
            pass

    # remove duplicates

    urls = list(dict.fromkeys(urls))

    combined_urls.append("|".join(urls))

df["image_url"] = combined_urls

# --------------------------------------------------
# convert image filenames to the same format
# --------------------------------------------------

def clean_filenames(x):

    if pd.isna(x):
        return ""

    try:

        lst = ast.literal_eval(x)

        if isinstance(lst, list):

            lst = [
                str(i).strip()
                for i in lst
                if str(i).strip()
            ]

            return "|".join(lst)

    except:
        pass

    return str(x).strip()

df["image_filename"] = df["image_filenames"].apply(clean_filenames)

# --------------------------------------------------
# real missing values
# --------------------------------------------------

print("image_url missing:")
print((df["image_url"].str.strip() == "").sum())

print()

print("image_filename missing:")
print((df["image_filename"].str.strip() == "").sum())